# LatticeFlow Studio from a notebook

The canonical soft-mode workflow, driven by `latticeflow_client`:

1. load cubic SrTiO3 and relax it under its symmetry (*is it relaxed?*)
2. phonons of the primitive cell, DOS and harmonic thermodynamics (*is it dynamically stable?*)
3. follow the imaginary R-point mode into the lower-symmetry phase
4. local structure: octahedral tilts in Glazer notation
5. compare the new phase with the cubic parent, check it, export it
6. a small batch screen

Start the backend first (`uvicorn app.main:app` in `backend/`, see the repo README) and install the client:
`pip install -e clients/python` (add `ase` / `pymatgen` for the conversion helpers).

If the studio UI is open on the same backend, note that it has its **own** session: this notebook drives a separate session (its own structure, undo history and model).

In [ ]:
from latticeflow_client import Studio, StudioError

studio = Studio("http://localhost:8000")   # connects; the server loads the SrTiO3 preset
studio.health()

## 1. Relax

`fix_symmetry=True` keeps Pm-3m (the cubic positions are fixed by symmetry, only `a` relaxes); `stress_tol_gpa` makes convergence require a stress-free cell, not only small forces.

In [ ]:
studio.load(preset="srtio3")
final = studio.relax(fmax=0.005, max_steps=300, fix_symmetry=True, stress_tol_gpa=0.05)
cubic = final["crystal"]
print(final["converged"], cubic["spacegroup_symbol"], round(cubic["lattice"]["a"], 4), "A",
      round(cubic["pressure_gpa"], 3), "GPa")

## 2. Phonons

`cell_mode="primitive"` gives the textbook dispersion (3 N_prim branches along the standard seekpath path). `dos=True` also requests the phonon DOS, the element-projected DOS and the harmonic free energy, entropy, heat capacity and zero-point energy.

In [ ]:
ph = studio.phonons(cell_mode="primitive", supercell_dim=[2, 2, 2], dos=True)
print("stable:", ph["is_dynamically_stable"], " min f:", ph["min_frequency_thz"], "THz",
      " supercell:", ph["supercell_dim"], f"({ph['min_image_distance']:.1f} A)")
[(m["label"], round(m["frequency_thz"], 2), m["ir_active"], m["raman_active"]) for m in ph["gamma_modes"] or []]

In [ ]:
# Optional: plot the dispersion (needs matplotlib)
import matplotlib.pyplot as plt

x = ph["kpath_distances"]
for branch in zip(*ph["frequencies_thz"]):
    plt.plot(x, branch, color="k", lw=0.8)
plt.xticks([x[l["index"]] for l in ph["kpath_labels"]],
           [l["label"].replace("\\Gamma", "Γ") for l in ph["kpath_labels"]])
plt.axhline(0, color="grey", lw=0.5); plt.ylabel("frequency (THz, negative = imaginary)")
plt.show()

Is the supercell large enough? `phonon_convergence()` reruns the analysis in the next larger supercell and reports the largest frequency change.

In [ ]:
conv = studio.phonon_convergence()["convergence"]
print(conv["supercell_dim_a"], "->", conv["supercell_dim_b"], " max |df| =", round(conv["max_abs_delta_thz"], 3),
      "THz  converged:", conv["converged"])

## 3. Follow the R-point instability

`soft_mode("R")` picks the lowest band at the labelled R point of the last analysis. The follow builds the supercell commensurate with q (2x2x2, 40 atoms), displaces the atoms along the eigenvector and relaxes. `on_frame` sees every relaxation frame; `studio.stop()` (e.g. from the callback) cancels the stream.

In [ ]:
q, band, f = studio.soft_mode("R")
print("R mode:", q, band, f, "THz")
energies = []
final = studio.follow_instability(q, band, amplitude=0.1, fmax=0.01, max_steps=400,
                                  on_frame=lambda fr: energies.append(fr["crystal"]["energy_per_atom"]))
distorted = final["crystal"]
print(distorted["spacegroup_symbol"], len(distorted["atoms"]), "atoms,",
      round((distorted["energy_per_atom"] - cubic["energy_per_atom"]) * 1000, 2), "meV/atom below cubic")
studio.refine(symprec=1e-3)["spacegroup_symbol"]

## 4. Local structure

Coordination, polyhedra, bond-valence sums and, for corner-sharing BX6 networks, the octahedral tilt system.

In [ ]:
local = studio.local_analysis()
local["octahedral_tilts"]["glazer"], local["octahedral_tilts"]["tilt_angles_deg"], local["global_instability_index"]

## 5. Compare, check and export

In [ ]:
cmp = studio.compare(cubic, studio.crystal)
{k: cmp[k] for k in ("spacegroup_a", "spacegroup_b", "group_subgroup", "volume_change_percent",
                     "energy_difference_mev_per_atom", "structures_match")}

In [ ]:
ph2 = studio.phonons(cell_mode="primitive", min_supercell_length=10)
el = studio.elastic_tensor(n_strains=2)   # relaxed-ion C_ij of the 40-atom cell: the slowest step (minutes)
print("dynamically stable:", ph2["is_dynamically_stable"], " Born stable:", el["born_stable"],
      " K_VRH =", round(el["k_vrh_gpa"]), "GPa  G_VRH =", round(el["g_vrh_gpa"]), "GPa")
cif = studio.export("cif", path="SrTiO3_I4mcm.cif")   # symmetrised CIF
atoms = studio.to_ase()                                # or studio.to_pymatgen()

In [ ]:
# E_hull against Materials Project reference phases relaxed with the same model (needs MP_API_KEY on the server)
try:
    print(studio.hull()["e_hull_ev"], "eV/atom above the hull")
except StudioError as exc:
    print(exc.code, exc.message)

The undo history holds every committed structure; `undo()` / `redo()` step through it.

In [ ]:
[(e["position"], e["label"], e["spacegroup_symbol"]) for e in studio.history["entries"]]

## 6. Batch screening

`screen()` runs generate (top_k) -> relax -> primitive phonons -> [hull] -> [elastic] per composition and keeps going when a row fails. Compositions are atoms per cell (Fm-3m rock salt needs the conventional `Na4Cl4`). The CLI equivalent is `latticeflow-screen compositions.csv --out results.csv --elastic`.

In [ ]:
from latticeflow_client.screen import screen

rows = screen([{"composition": "SrTiO3", "target_spacegroup": 221}, {"composition": "Si8", "target_spacegroup": 227}],
              studio=studio, top_k=2, elastic=True)
[(r["composition"], r.get("candidate_rank"), r.get("spacegroup"), r.get("min_frequency_thz"), r.get("k_vrh_gpa"),
  r.get("error")) for r in rows]

In [ ]:
studio.close()